# Keithley 2460 guarded 100 Ω resistor smoke test

This notebook reproduces the first bounded source/measure test on the Model 2460. It is deliberately specific to a **nominal 100 Ω resistor on the front terminals with four-wire sense leads**. Do not run it with a coin cell, battery, capacitor, unknown load, or a second controller connected.

Before running the source cell:

1. Disconnect every battery or other source.
2. Connect the 100 Ω resistor: FORCE HI and SENSE HI at one end; FORCE LO and SENSE LO at the other.
3. Select the front terminals and confirm the physical OUTPUT indicator is off.
4. Set `CONFIRM_FRONT_4WIRE_100_OHM` to `True`.

The test first checks output OFF and front-terminal routing. It then sources 100 µA and, only after a plausible 80–120 Ω result, 1 mA. The voltage limit is 0.2 V, the lowest supported current-source limit on this instrument. A `finally` block switches output OFF and programs 0 A. If communication fails while output might be on, use the front-panel OUTPUT control immediately and inspect the instrument state before reconnecting.

The resistor is not a calibration standard. This is a plausibility and shutdown smoke test, not an accuracy or hardware-acceptance test. Commands were reviewed against the official [Model 2460 Reference Manual, revision C](https://download.tek.com/manual/2460-901-01C_Sept_2019_Ref.pdf).

## Environment

From the repository root, run `uv sync --locked --python 3.12 --extra visa`, then select the repository `.venv` as the notebook kernel.

In [ ]:
import json
import math
from datetime import UTC, datetime

from ophyd_electrochemistry.keithley.k2460.transport import (
    PyVisaKeithley2460Transport,
    VisaTransportConfig,
)

RESOURCE = "TCPIP0::169.254.77.125::5025::SOCKET"
CONFIRM_FRONT_4WIRE_100_OHM = False  # Change only after the physical checklist.
CURRENT_POINTS_A = (100e-6, 1e-3)
VOLTAGE_LIMIT_V = 0.2
PLAUSIBLE_RESISTANCE_OHM = (80.0, 120.0)

In [ ]:
def finite_number(reply: str, label: str) -> float:
    try:
        value = float(reply.strip())
    except ValueError as exc:
        raise RuntimeError(f"{label}: nonnumeric reply {reply!r}") from exc
    if not math.isfinite(value):
        raise RuntimeError(f"{label}: nonfinite reply {reply!r}")
    return value


def require_token(actual: str, allowed: tuple[str, ...], label: str) -> None:
    if actual not in allowed:
        raise RuntimeError(f"{label}: expected one of {allowed}, received {actual!r}")

## Run the bounded test

This cell does not reset the instrument or recall saved settings. It verifies all safety-relevant configuration before enabling output. Output is enabled only around each single reading.

In [ ]:
if not CONFIRM_FRONT_4WIRE_100_OHM:
    raise RuntimeError(
        "Physical setup not confirmed. Complete the checklist and set "
        "CONFIRM_FRONT_4WIRE_100_OHM = True."
    )

config = VisaTransportConfig(
    resource_name=RESOURCE,
    backend="@py",
    required_language="SCPI",
    io_timeout_ms=5_000,
)
report = {
    "schema": "k2460-resistor-smoke-v1",
    "observed_at_utc": datetime.now(UTC).isoformat(),
    "resource_name": RESOURCE,
    "nominal_resistance_ohm": 100.0,
    "voltage_limit_v": VOLTAGE_LIMIT_V,
    "points": [],
}
instrument = PyVisaKeithley2460Transport(config)

try:
    instrument.connect()
    report["identity"] = instrument.identity.raw
    require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "preflight output")
    require_token(instrument.query(":ROUTe:TERMinals?"), ("FRON", "FRONT"), "terminals")

    for command in (
        ":FORMat:DATA ASCii",
        ":SOURce:FUNCtion CURRent",
        ":SOURce:CURRent 0",
        f":SOURce:CURRent:VLIMit {VOLTAGE_LIMIT_V}",
        ":SOURce:CURRent:READ:BACK ON",
        ':SENSe:FUNCtion "VOLTage"',
        ":SENSe:VOLTage:RSENse ON",
        ":SENSe:VOLTage:RANGe 0.2",
        ":SENSe:VOLTage:NPLCycles 1",
        ":SENSe:VOLTage:AVERage OFF",
        ":SENSe:VOLTage:RELative:STATe OFF",
        ":SENSe:COUNt 1",
    ):
        instrument.write(command)

    verified = {
        "output": instrument.query(":OUTPut:STATe?"),
        "terminals": instrument.query(":ROUTe:TERMinals?"),
        "source_function": instrument.query(":SOURce:FUNCtion?"),
        "voltage_limit_v": instrument.query(":SOURce:CURRent:VLIMit?"),
        "source_readback": instrument.query(":SOURce:CURRent:READ:BACK?"),
        "measure_function": instrument.query(":SENSe:FUNCtion?"),
        "remote_sense": instrument.query(":SENSe:VOLTage:RSENse?"),
        "voltage_range_v": instrument.query(":SENSe:VOLTage:RANGe?"),
        "voltage_nplc": instrument.query(":SENSe:VOLTage:NPLCycles?"),
        "filter": instrument.query(":SENSe:VOLTage:AVERage?"),
        "relative": instrument.query(":SENSe:VOLTage:RELative:STATe?"),
        "count": instrument.query(":SENSe:COUNt?"),
    }
    report["verified_setup"] = verified
    require_token(verified["output"], ("0", "OFF"), "configured output")
    require_token(verified["terminals"], ("FRON", "FRONT"), "configured terminals")
    require_token(verified["source_function"], ("CURR", "CURRENT"), "source function")
    require_token(
        verified["measure_function"], ('"VOLT:DC"', '"VOLT"', "VOLT:DC", "VOLT"), "measure function"
    )
    require_token(verified["remote_sense"], ("1", "ON"), "remote sense")
    require_token(verified["source_readback"], ("1", "ON"), "source readback")
    require_token(verified["filter"], ("0", "OFF"), "filter")
    require_token(verified["relative"], ("0", "OFF"), "relative offset")
    if abs(finite_number(verified["voltage_limit_v"], "voltage limit") - 0.2) > 1e-9:
        raise RuntimeError("voltage limit verification failed")
    if abs(finite_number(verified["voltage_range_v"], "voltage range") - 0.2) > 1e-9:
        raise RuntimeError("voltage range verification failed")
    if finite_number(verified["count"], "measure count") != 1:
        raise RuntimeError("measure count verification failed")

    for requested_current_a in CURRENT_POINTS_A:
        instrument.write(f":SOURce:CURRent:RANGe {requested_current_a:.12g}")
        instrument.write(f":SOURce:CURRent {requested_current_a:.12g}")
        require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "pre-point output")
        programmed_a = finite_number(instrument.query(":SOURce:CURRent?"), "programmed current")
        source_range_a = finite_number(instrument.query(":SOURce:CURRent:RANGe?"), "source range")
        if abs(programmed_a - requested_current_a) > max(1e-12, requested_current_a * 1e-6):
            raise RuntimeError("programmed current verification failed")
        if source_range_a + 1e-15 < abs(requested_current_a):
            raise RuntimeError("source range is smaller than requested current")

        try:
            instrument.write(":OUTPut:STATe ON")
            require_token(instrument.query(":OUTPut:STATe?"), ("1", "ON"), "energized output")
            raw = instrument.query(':READ? "defbuffer1", SOURce, READing')
        finally:
            instrument.write(":OUTPut:STATe OFF")
            require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "post-point output")

        fields = [field.strip() for field in raw.split(",")]
        if len(fields) != 2:
            raise RuntimeError(f"expected source/read pair, received {raw!r}")
        measured_current_a = finite_number(fields[0], "source readback")
        measured_voltage_v = finite_number(fields[1], "voltage reading")
        limit_tripped = instrument.query(":SOURce:CURRent:VLIMit:TRIPped?")
        require_token(limit_tripped, ("0", "1"), "voltage-limit state")
        resistance_ohm = measured_voltage_v / measured_current_a
        point = {
            "requested_current_a": requested_current_a,
            "source_range_a": source_range_a,
            "measured_current_a": measured_current_a,
            "measured_voltage_v": measured_voltage_v,
            "resistance_ohm": resistance_ohm,
            "voltage_limit_tripped": limit_tripped == "1",
            "output_off_confirmed": True,
        }
        report["points"].append(point)
        low, high = PLAUSIBLE_RESISTANCE_OHM
        if not (low <= resistance_ohm <= high) or limit_tripped == "1":
            raise RuntimeError(f"plausibility gate failed: {point}")
        instrument.write(":SOURce:CURRent 0")
finally:
    cleanup_error = None
    if instrument.connected:
        try:
            instrument.write(":OUTPut:STATe OFF")
            require_token(instrument.query(":OUTPut:STATe?"), ("0", "OFF"), "final output")
            instrument.write(":SOURce:CURRent 0")
            report["final_output_enabled"] = False
            report["final_programmed_current_a"] = finite_number(
                instrument.query(":SOURce:CURRent?"), "final programmed current"
            )
        except BaseException as exc:
            cleanup_error = exc
    try:
        instrument.close()
    except BaseException as exc:
        cleanup_error = cleanup_error or exc
    if cleanup_error is not None:
        raise RuntimeError(
            f"cleanup failed; inspect the front panel: {cleanup_error}"
        ) from cleanup_error

print(json.dumps(report, indent=2))

## Observed result on 2026-10-05

On instrument serial `04686198`, firmware `1.7.16a`, the test measured:

- 99.99917 µA source readback and 10.27817 mV: 102.7826 Ω.
- 0.9999969 mA source readback and 102.5312 mV: 102.5315 Ω.
- No voltage-limit trip at either point.
- Output OFF after both points, final programmed current 0 A.
- An independent query-only session confirmed output state `0`.

The machine-readable record is `docs/evidence/k2460-2026-10-05-resistor-smoke.json`.